# Validation des chiffres — AlphaForge Témoin
**Responsable : Habib (Lead Data Scientist)** · Datathon PolyFinances 2026

**But :** prouver que les reçus de la Machine à remonter le temps disent vrai.

**Méthode :** je recalcule chaque chiffre **sans utiliser le code de l'équipe** (`machine.py`), avec une méthode écrite séparément, puis je compare mon résultat au reçu. Si les deux méthodes donnent le même chiffre, le reçu est validé.

Lancer depuis la racine du repo : `jupyter notebook notebooks/validation_habib.ipynb`, puis **Run All**.

In [1]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd

os.environ["ALPHAFORGE_HORS_LIGNE"] = "1"
RACINE = Path.cwd() if (Path.cwd() / "alphaforge").exists() else Path.cwd().parent
sys.path.insert(0, str(RACINE))
DATA = RACINE / "data"
pd.set_option("display.width", 160)
print("Repo :", RACINE)

Repo : /home/claude/new/alphaforge-ai-equipe25-main


## 1. Recalcul indépendant

Mes choix, différents de ceux de `machine.py` pour que la comparaison ait du sens :
- **Conversion en CAD** avec `merge_asof` (je prends le dernier taux USD/CAD publié **au plus tard** ce jour-là). `machine.py` fait un `reindex` + `ffill`.
- **Volatilité** (ampleur des variations de prix, sur un an) : écart-type des rendements journaliers en log × √(nombre de jours de cotation par an), 252 pour les actions et FNB, 365 pour la crypto.
- **Perte maximale** (pire chute entre un sommet et le creux suivant) : calculée avec une boucle simple, sans `cummax`.
- **Résultat 2022** : dernier prix connu au 31 décembre 2022 / dernier prix connu au 31 décembre 2021 − 1.

In [2]:
prix = pd.read_csv(DATA / "prices_daily.csv", parse_dates=["date"])
fx = pd.read_csv(DATA / "fx_usdcad_daily.csv", parse_dates=["date"]).sort_values("date")

def serie_cad_independante(ticker):
    s = prix[prix.ticker == ticker].sort_values("date")[["date", "adj_close", "currency"]]
    if s.currency.iloc[0] == "USD":
        s = pd.merge_asof(s, fx[["date", "usdcad"]], on="date", direction="backward")
        s["usdcad"] = s["usdcad"].fillna(fx.usdcad.iloc[0])
        s["cad"] = s.adj_close * s.usdcad
    else:
        s["cad"] = s.adj_close
    return s.set_index("date")["cad"]

def volatilite(s, jours_par_an):
    r = np.log(s.values[1:] / s.values[:-1])
    return r.std(ddof=1) * np.sqrt(jours_par_an) * 100

def perte_maximale(s):
    sommet, pire = s.iloc[0], 0.0
    for x in s:
        sommet = max(sommet, x)
        pire = min(pire, x / sommet - 1)
    return pire * 100

def resultat_annee(s, annee):
    return (s.loc[:f"{annee}-12-31"].iloc[-1] / s.loc[:f"{annee - 1}-12-31"].iloc[-1] - 1) * 100

## 2. Comparaison avec les reçus de l'équipe

In [3]:
from alphaforge.machine import rejouer
from alphaforge.recus import Registre

ACTIFS = ["TSLA", "SPY", "XIU.TO", "ZST.TO", "BTC-USD", "XOM"]
TOLERANCE = 0.05  # points de pourcentage

lignes = []
for t in ACTIFS:
    s = serie_cad_independante(t)
    n = 365 if t.endswith("-USD") else 252
    moi = {"vol": volatilite(s, n), "perte_max": perte_maximale(s), "an2022": resultat_annee(s, 2022)}
    reg = Registre()
    res = rejouer({t: 1.0}, 10000, 4, reg, t)
    for cle, nom in [("vol", "Volatilité annuelle"), ("perte_max", "Perte maximale"), ("an2022", "Résultat 2022")]:
        recu = res["recus"][cle]
        ecart = abs(moi[cle] - recu.valeur)
        lignes.append({"actif": t, "chiffre": nom, "recalcul manuel (%)": round(moi[cle], 2),
                       "reçu (%)": round(recu.valeur, 2), "écart": round(ecart, 4),
                       "verdict": "VALIDÉ" if ecart <= TOLERANCE else "À EXAMINER"})

tableau = pd.DataFrame(lignes)
print(tableau.to_string(index=False))
print("\n", (tableau.verdict == "VALIDÉ").sum(), "/", len(tableau), "chiffres validés")
(RACINE / "docs").mkdir(exist_ok=True)
tableau.to_csv(RACINE / "docs" / "validation_tableau.csv", index=False)

  actif             chiffre  recalcul manuel (%)  reçu (%)  écart verdict
   TSLA Volatilité annuelle                58.79     58.79    0.0  VALIDÉ
   TSLA      Perte maximale               -70.88    -70.88    0.0  VALIDÉ
   TSLA       Résultat 2022               -62.64    -62.64    0.0  VALIDÉ
    SPY Volatilité annuelle                16.65     16.65    0.0  VALIDÉ
    SPY      Perte maximale               -22.25    -22.25    0.0  VALIDÉ
    SPY       Résultat 2022               -12.59    -12.59    0.0  VALIDÉ
 XIU.TO Volatilité annuelle                13.07     13.07    0.0  VALIDÉ
 XIU.TO      Perte maximale               -16.36    -16.36    0.0  VALIDÉ
 XIU.TO       Résultat 2022                -6.35     -6.35    0.0  VALIDÉ
 ZST.TO Volatilité annuelle                 0.56      0.56    0.0  VALIDÉ
 ZST.TO      Perte maximale                -0.43     -0.43    0.0  VALIDÉ
 ZST.TO       Résultat 2022                 1.22      1.22    0.0  VALIDÉ
BTC-USD Volatilité annuelle           

## 3. Exxon : pourquoi « 30 000 $ → 106 500 $ » ?

Le chiffre paraît énorme (×3,5 en 5 ans). Je le décompose en trois effets pour vérifier qu'il vient des données et non d'une erreur.

In [4]:
x = prix[prix.ticker == "XOM"].set_index("date")
f = fx.set_index("date").usdcad
effet_prix = x.close.iloc[-1] / x.close.iloc[0]
effet_dividendes = (x.adj_close.iloc[-1] / x.adj_close.iloc[0]) / effet_prix
effet_change = f.iloc[-1] / f.iloc[0]
total = effet_prix * effet_dividendes * effet_change
print(f"Prix de l'action (USD)        : {x.close.iloc[0]:.2f} → {x.close.iloc[-1]:.2f}   × {effet_prix:.3f}")
print(f"Dividendes réinvestis          : × {effet_dividendes:.3f}")
print(f"Dollar canadien (USD/CAD)      : {f.iloc[0]:.4f} → {f.iloc[-1]:.4f}   × {effet_change:.3f}")
print(f"Total                          : × {total:.3f}  → 30 000 $ deviennent {30000 * total:,.0f} $".replace(",", " "))

Prix de l'action (USD)        : 63.82 → 169.78   × 2.660
Dividendes réinvestis          : × 1.176
Dollar canadien (USD/CAD)      : 1.2565 → 1.4255   × 1.135
Total                          : × 3.549  → 30 000 $ deviennent 106 458 $


**Conclusion :** le chiffre est **juste**. Il vient de trois effets réels qui se multiplient : la hausse du prix d'Exxon, les dividendes réinvestis (`adj_close`) et la baisse du dollar canadien, qui gonfle la valeur en CAD d'un actif américain. À dire dans la documentation : « montant en CAD, dividendes réinvestis ».

## 4. Pièges des données

In [5]:
print("— Piège 1 : CNQ.TO, faux « fractionnement » de 1,021 le 2022-08-22")
c = prix[prix.ticker == "CNQ.TO"].set_index("date").loc["2022-08-18":"2022-08-24", ["close", "adj_close"]]
c["var close %"] = c.close.pct_change() * 100
c["var adj_close %"] = c.adj_close.pct_change() * 100
print(c.round(3))
div = 0.75 / c.close.loc["2022-08-19"] * 100
print(f"Saut de adj_close le 22 août : {c.loc['2022-08-22','var adj_close %'] - c.loc['2022-08-22','var close %']:.2f} points"
      f" ≈ dividende de 0,75 $ / prix ({div:.2f} %).")
print("=> Le facteur 1,021 n'est PAS appliqué une 2e fois : adj_close ne compte le dividende qu'une fois.")
print("   CNQ n'est utilisé dans aucun scénario de démo. Impact : nul sur la démo.")

— Piège 1 : CNQ.TO, faux « fractionnement » de 1,021 le 2022-08-22
             close  adj_close  var close %  var adj_close %
date                                                       
2022-08-18  35.465     28.518          NaN              NaN
2022-08-19  35.451     28.506       -0.041           -0.041
2022-08-22  35.665     29.299        0.605            2.779
2022-08-23  36.335     29.849        1.879            1.879
2022-08-24  36.680     30.132        0.950            0.950
Saut de adj_close le 22 août : 2.17 points ≈ dividende de 0,75 $ / prix (2.12 %).
=> Le facteur 1,021 n'est PAS appliqué une 2e fois : adj_close ne compte le dividende qu'une fois.
   CNQ n'est utilisé dans aucun scénario de démo. Impact : nul sur la démo.


In [6]:
print("— Piège 2 : la crypto se négocie 7 jours sur 7")
print(prix.groupby("ticker").size().loc[["BTC-USD", "ETH-USD", "SPY", "XIU.TO"]].to_string())

from alphaforge.machine import serie_portefeuille
v = serie_portefeuille({"SPY": 0.85, "BTC-USD": 0.15}, 60000)
r = np.log(v).diff().dropna()
obs_par_an = len(r) / ((v.index[-1] - v.index[0]).days / 365.25)
print(f"\nPortefeuille 85 % SPY + 15 % BTC : {obs_par_an:.0f} observations par an (calendrier 7/7)")
print(f"  Volatilité affichée par machine.py (× √252) : {r.std() * np.sqrt(252) * 100:.1f} %")
print(f"  Volatilité corrigée (× √{obs_par_an:.0f})            : {r.std() * np.sqrt(obs_par_an) * 100:.1f} %")
print("=> ÉCART : pour un portefeuille qui MÉLANGE crypto et actions, machine.py annualise avec 252 jours")
print("   alors que la série a ~365 points par an. La volatilité est sous-estimée d'environ 17 %.")
print("   Les scénarios de démo (un seul actif) ne sont PAS touchés : BTC seul utilise bien 365.")

— Piège 2 : la crypto se négocie 7 jours sur 7
ticker
BTC-USD    1827
ETH-USD    1827
SPY        1255
XIU.TO     1255

Portefeuille 85 % SPY + 15 % BTC : 365 observations par an (calendrier 7/7)
  Volatilité affichée par machine.py (× √252) : 14.6 %
  Volatilité corrigée (× √365)            : 17.6 %
=> ÉCART : pour un portefeuille qui MÉLANGE crypto et actions, machine.py annualise avec 252 jours
   alors que la série a ~365 points par an. La volatilité est sous-estimée d'environ 17 %.
   Les scénarios de démo (un seul actif) ne sont PAS touchés : BTC seul utilise bien 365.


In [7]:
print("— Piège 3 : conversion USD → CAD")
fx_jours = set(fx.date)
us = prix[prix.currency == "USD"]
sans_taux = us[~us.date.isin(fx_jours)].date.nunique()
print(f"Jours de cotation américains sans taux de la Banque du Canada (fériés canadiens, fins de semaine crypto) : {sans_taux}")
print("=> machine.py reprend le dernier taux connu (ffill). Mon recalcul fait pareil avec merge_asof : mêmes chiffres.")
t = prix[(prix.ticker == "SPY") & (prix.date == "2022-12-30")].iloc[0]
taux = fx.set_index("date").usdcad.loc["2022-12-30"]
from alphaforge.machine import serie_cad
print(f"Contrôle ponctuel SPY le 2022-12-30 : {t.adj_close:.4f} USD × {taux} = {t.adj_close * taux:.4f} CAD ;"
      f" machine.py : {serie_cad('SPY').loc['2022-12-30']:.4f} CAD")
print("Les titres .TO sont déjà en CAD : aucun taux appliqué (vérifié dans machine.py, fx = 1,0).")

print("\n— Piège 4 : ZST.TO a une ligne à volume nul le 2022-07-06")
print(prix[(prix.ticker == "ZST.TO") & (prix.volume == 0)][["date", "close", "adj_close", "volume"]].to_string(index=False))
print("=> Prix cohérent avec les jours voisins : ligne conservée, aucun effet visible.")

— Piège 3 : conversion USD → CAD
Jours de cotation américains sans taux de la Banque du Canada (fériés canadiens, fins de semaine crypto) : 582
=> machine.py reprend le dernier taux connu (ffill). Mon recalcul fait pareil avec merge_asof : mêmes chiffres.
Contrôle ponctuel SPY le 2022-12-30 : 364.7676 USD × 1.3544 = 494.0412 CAD ; machine.py : 494.0412 CAD
Les titres .TO sont déjà en CAD : aucun taux appliqué (vérifié dans machine.py, fx = 1,0).

— Piège 4 : ZST.TO a une ligne à volume nul le 2022-07-06
      date     close  adj_close  volume
2022-07-06 48.810001  41.455101       0
=> Prix cohérent avec les jours voisins : ligne conservée, aucun effet visible.


## 5. Test du Vérificateur et du garde-fou bloquant

In [8]:
import importlib, scripts.test_verificateur as tv
importlib.reload(tv)
tv.main()

resultat   score  attendu  cas
OK         100.0  accepte  chiffre exact, recu exact : Tesla a perdu −71 % [#R-008].
OK         100.0  accepte  « perdu » sans signe moins : accepte : Tesla a perdu 71 % [#R-008].
OK         100.0  accepte  volatilite exacte : Sa volatilite annuelle est de 59 % [#R-007].
OK         100.0  accepte  montant arrondi : Au pire moment, le placement valait 20 500 $ [#R-009].
OK         100.0  accepte  baisse annoncee en mots : En 2022, Tesla a recule de 63 % [#R-010].
OK          50.0  refuse   bon chiffre, MAUVAIS recu : Tesla a perdu −63 % [#R-008].
OK           0.0  refuse   chiffre invente : Tesla a perdu −80 % [#R-008].
OK           0.0  refuse   chiffre invente, sans recu : Tesla a perdu −80 %.
OK          90.0  partiel  chiffre exact mais NON CITE (regle de Celestine : 90 %) : Tesla a perdu −71 %.
OK          50.0  refuse   recu qui n'existe pas : Tesla a perdu −71 % [#R-999].
OK           0.0  refuse   montant faux : Au pire moment, le placement valait 

0

Le test contient les cas de la roadmap (chiffre faux, mauvais reçu, reçu inexistant) et **4 cas de sens inversé** (« Tesla a gagné 71 % » alors que c'est une perte). Sans le correctif du 14 novembre, ces 4 cas sont acceptés : voir `docs/validation_habib.md`. Le test vérifie aussi la règle « citation obligatoire » de Celestine (un chiffre juste mais sans numéro de reçu plafonne à 90 %).

## 6. Constats pour le pitch

In [9]:
def rendements_cad(t):
    return np.log(serie_cad_independante(t)).diff()

spy = rendements_cad("SPY")
print("Corrélation des rendements journaliers avec SPY (en CAD, jours communs) :")
for t, quoi in [("QQQ", "technologie (Nasdaq-100)"), ("ESGV", "FNB filtré ESG"), ("GLD", "or")]:
    rho = pd.concat([spy, rendements_cad(t)], axis=1).dropna().corr().iloc[0, 1]
    print(f"  {t:5} ({quoi:24}) : {rho:.2f}")

print("\nRésultat sur l'année 2022, en CAD, dividendes réinvestis :")
for t in ["SPY", "XIU.TO", "QQQ", "ESGV", "XBB.TO", "AGG", "TLT", "GLD", "XOM", "TSLA", "BTC-USD"]:
    print(f"  {t:8} : {resultat_annee(serie_cad_independante(t), 2022):+7.1f} %")

Corrélation des rendements journaliers avec SPY (en CAD, jours communs) :
  QQQ   (technologie (Nasdaq-100)) : 0.95
  ESGV  (FNB filtré ESG          ) : 0.99
  GLD   (or                      ) : 0.09

Résultat sur l'année 2022, en CAD, dividendes réinvestis :
  SPY      :   -12.6 %
  XIU.TO   :    -6.3 %
  QQQ      :   -28.0 %
  ESGV     :   -18.9 %
  XBB.TO   :   -11.7 %
  AGG      :    -7.1 %
  TLT      :   -26.5 %
  GLD      :    +6.0 %
  XOM      :  +100.2 %
  TSLA     :   -62.6 %
  BTC-USD  :   -61.8 %


**Lecture pour le pitch :**
- **ESGV bouge presque comme SPY** : choisir un fonds ESG ne protège pas d'une baisse du marché. Utile pour Mateo.
- **L'or (GLD) est peu corrélé à SPY** : c'est un vrai diversifiant (un placement qui ne baisse pas forcément en même temps que les actions).
- **En 2022, les obligations (XBB, AGG, TLT) ont baissé en même temps que les actions** : c'est la preuve, dans nos propres données, du piège que vit Robert.
- **XOM a fortement monté en 2022** : un placement qui « marche » peut quand même être exclu par les valeurs de Mateo.

*Le passé ne prédit pas l'avenir : ce sont des constats sur la période 2021-2026, pas des prévisions.*